In [1]:
from model_ranking import BBBC039TargetConfig
from model_ranking import (
    HammingDistanceEval
)
from model_ranking import (
    InputConsistencyPatchwisePseudoLabeler,
)
from model_ranking import (
    get_model_path,
)
from model_ranking import (
    add_device_to_config,
)
import torch
from pytorch3dunet.datasets.utils import get_test_loaders, get_train_loaders
from pytorch3dunet.datasets.dsb import TIF_txt_Dataset
from typing import Literal, List, Dict, Any
from pytorch3dunet.augment.transforms import (
    Transformer,
)
from pytorch3dunet.unet3d.model import get_model
from pytorch3dunet.unet3d.utils import load_checkpoint

In [2]:
target_cfg = BBBC039TargetConfig()
pred_loader = target_cfg.loader

In [3]:
pred_loader_cfg = pred_loader.create_config(
    output_dir="out_path",
    data_base_path="/scratch/talks/data",
).model_dump()
config = {"loaders": pred_loader_cfg}
config = add_device_to_config(config)

In [4]:
transformer_config: Dict[str, List[Any]] = {
    "raw": [
        {"name": "AdditiveGaussianNoise", "execution_probability": 1, "scale": [0.1, 0.2]},
    ]
}

stats = {
        'pmin': None,
        'pmax': None,
        'mean': None,
        'std': None,
        'percentile_min': None,
        'percentile_max': None,
    }


In [5]:
transformer = Transformer(transformer_config, stats)

In [6]:
model_cfg = {
    "name": "UNet2D",
    "in_channels": 1,
    "out_channels": 1,
    "layer_order": "bcr",
    "f_maps": [32, 64, 128],
    "final_sigmoid": True,
    "feature_return": False,
    "is_segmentation": True,
    "feature_perturbation": None,
}
model_path = get_model_path("BBBC039", "BC_model", "/g/kreshuk/talks/Models")
model = get_model(model_cfg)
load_checkpoint(model_path, model)
model = model.cuda()

/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(checkpoint_path, map_location="cpu")


In [7]:
consis_metric = HammingDistanceEval(0.5)

In [8]:
pseudo_labeler = InputConsistencyPatchwisePseudoLabeler(
    transformer=transformer,
    consistency_metric=consis_metric,
    foreground_threshold=0.5,
    consistency_threshold=0.5,
)

In [ ]:
for test_loader in get_test_loaders(config):
    img, _ = next(iter(test_loader))
    print(img.shape)
    img = torch.squeeze(img, dim=-3)
    img = img.cuda()
    pseudo_labels, label_mask = pseudo_labeler(model, img)
    break

2025-05-15 14:02:13,411 [MainThread] INFO Dataset - Creating test set loaders...
2025-05-15 14:02:14,305 [MainThread] INFO Dataset - Number of workers for the dataloader: 8
2025-05-15 14:02:14,306 [MainThread] INFO Dataset - 8 GPUs available. Using batch_size = 8 * 2
2025-05-15 14:02:14,306 [MainThread] INFO Dataset - Batch size for dataloader: 16
2025-05-15 14:02:14,307 [MainThread] INFO Dataset - Loading test set from: /scratch/talks/data/BBBC039/images...


In [3]:
import numpy as np
a = np.zeros((2,1,2,128,128))
b = np.ones((2,128,128))
a[0] = b
print(a[0])
print(a.shape)
print(a[1])

[[[[1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]
   ...
   [1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]]

  [[1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]
   ...
   [1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]
   [1. 1. 1. ... 1. 1. 1.]]]]
(2, 1, 2, 128, 128)
[[[[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]]]
